Will again just be using 'temp' as my variable name, so be aware, you are on the NITRATE dataset script, not Temp. 
The variable column is labeled as surf temp, stands in for surf_NO3 (umol/L).
I am sorry for the confusion. I copied the code I used for getting temperature metrics and replacing these variable names
would have taken forever... 

In [1]:
import h5pyd
import h5py
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import haversine
from haversine import haversine, Unit
import numpy as np
from datetime import datetime

In [2]:
sites = pd.read_csv('C:/Users/sarah/Documents/GitHub/ORKA_StatusReport_FollowupPaper/StatusReport_FollowupPaper_Data/MasterSiteList_FollowupPaper.csv')
sites = sites.iloc[0:15,0:8]

In [10]:
#8 - 9 minutes
#Combine the two datasets together. Realized that the initial ROMS dataframe was cropped just a little bit too far west and we needed to add back
#in a bit of data from further east in the larger dataset. Ended up just combining the two portions of raw data together in this step instead of 
#making a new clean one for the sake of efficiency and dealing with these huge freaking datasets. Not pretty, but it worked. 
#temp is 235,526,278 rows long
temp = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles_NO3/surface_NO3_all.csv")
#temp_east is 10,733,967 rows long
temp_east = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles_NO3/surface_NO3_east_all.csv")
temp = pd.concat([temp,temp_east])
temp = temp.dropna()
print("Number of time steps is", temp.ocean_time.nunique())
print("Lat/lon combos is", len(temp[['lon_rho','lat_rho']].drop_duplicates()))
#drop 2024 values and reset index
temp.ocean_time = pd.to_datetime(temp.ocean_time)
temp = temp.loc[temp.ocean_time < "2023-12-31"].reset_index()

Number of time steps is 4383
Lat/lon combos is 48278


In [13]:
%%time
#Ok here I'm finding which ROMS grid cell is closest to each site 
#then pulling out the ROMS data for each nearest grid cell and putting it into a dataframe called 'temp_forsites' 
lat = sites.SiteLatitude[0]
lon = sites.SiteLongitude[0]
site = (lat, lon)
site_rounded = (round(lat,2), round(lon, 2))
temp_sub = temp.loc[(temp.lat_rho < site[0] +0.01) & (temp.lat_rho > site[0]-0.01)]
def hav_dist(row):
    site2 = (row["lat_rho"],row["lon_rho"])
    return haversine(site,site2)
dist = temp_sub.apply(hav_dist, axis = 1)
index1 = dist[dist == dist.min()].index
temp_forsites = temp.iloc[index1.values[:],]

for i in range(1,len(sites)): 
    #print(i, "has the latitude", sites.SiteLatitude[i])
    site = (sites.SiteLatitude[i], sites.SiteLongitude[i])
    temp_sub = temp.loc[(temp.lat_rho < site[0] +0.01) & (temp.lat_rho > site[0]-0.01)]
    dist = temp_sub.apply(hav_dist, axis = 1)
    index1 = dist[dist == dist.min()].index
    closest = temp.iloc[index1.values[:],]
    print(i)
    print(closest.iloc[1,6])
    temp_forsites = pd.concat([temp_forsites,closest])
    print(i," is finished, hooray!")
temp_forsites.ocean_time = pd.to_datetime(temp_forsites.ocean_time)

1
45.21331748549788
1  is finished, hooray!
2
44.78383099999047
2  is finished, hooray!
3
44.74972204231666
3  is finished, hooray!
4
43.342748340363855
4  is finished, hooray!
5
43.32052277193761
5  is finished, hooray!
6
43.29821697682596
6  is finished, hooray!
7
42.83479975382984
7  is finished, hooray!
8
42.78659473126096
8  is finished, hooray!
9
42.73804107337505
9  is finished, hooray!
10
42.70139552190807
10  is finished, hooray!
11
42.66455137467071
11  is finished, hooray!
12
42.43925529923458
12  is finished, hooray!
13
42.06055447045627
13  is finished, hooray!
14
42.03370062156226
14  is finished, hooray!
CPU times: total: 6min 1s
Wall time: 5min 50s


In [14]:
#Check which grid cells from ROMs model are pulled out to correspond with which sites
temp_forsites[["lat_rho","lon_rho"]].drop_duplicates()

,lat_rho,lon_rho
184764617,45.335118,-123.981283
184764509,45.213317,-123.988016
21078,44.783831,-124.077410
20654,44.749722,-124.070370
6739,43.342748,-124.374920
6606,43.320523,-124.408119
6479,43.298217,-124.399769
4073,42.834800,-124.582252
3882,42.786595,-124.591327
3692,42.738041,-124.519727


In [15]:
#Calculate 10th percentile of nitrate at each site
temp90 = temp_forsites.groupby("lat_rho").surf_temp.quantile(q = 0.1).reset_index()
temp90
#Adding this new nitrate metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["P10_NO3"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_NO3
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,1.959780
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,1.980009
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,2.470371
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,2.549008
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,3.473037
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,3.545663
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,4.526276
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,5.512704
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,5.833533
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,6.086707


In [16]:
#Mean nitrate at each site
temp90 = temp_forsites.groupby("lat_rho").surf_temp.mean().reset_index()
temp90
#Adding this new nitrate metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["Mean_NO3"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_NO3,Mean_NO3
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,1.959780,12.215722
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,1.980009,11.560737
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,2.470371,14.330551
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,2.549008,15.126274
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,3.473037,14.665780
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,3.545663,14.368735
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,4.526276,15.796427
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,5.512704,17.611699
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,5.833533,17.763879
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,6.086707,18.790775


In [17]:

#90th percentile of nitrate at each site
temp90 = temp_forsites.groupby("lat_rho").surf_temp.quantile(q = 0.9).reset_index()
temp90
#Adding this new nitrate metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["P90_NO3"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_NO3,Mean_NO3,P90_NO3
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,1.959780,12.215722,27.606370
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,1.980009,11.560737,26.331103
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,2.470371,14.330551,32.062062
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,2.549008,15.126274,33.368498
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,3.473037,14.665780,30.624957
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,3.545663,14.368735,29.879866
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,4.526276,15.796427,31.443690
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,5.512704,17.611699,33.152401
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,5.833533,17.763879,33.093110
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,6.086707,18.790775,34.137105


In [18]:
#10th percentile of growing season
temp_growing = temp_forsites.loc[(temp_forsites.ocean_time.dt.month>=4) & (temp_forsites.ocean_time.dt.month<=9)]
temp90 = temp_growing.groupby("lat_rho").surf_temp.quantile(q = 0.1).reset_index()
temp90
#Adding this new nitrate metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["P10_NO3_growing"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_NO3,Mean_NO3,P90_NO3,P10_NO3_growing
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,1.959780,12.215722,27.606370,1.247808
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,1.980009,11.560737,26.331103,1.072389
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,2.470371,14.330551,32.062062,1.715417
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,2.549008,15.126274,33.368498,1.709247
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,3.473037,14.665780,30.624957,1.826484
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,3.545663,14.368735,29.879866,1.907076
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,4.526276,15.796427,31.443690,3.330555
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,5.512704,17.611699,33.152401,4.562867
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,5.833533,17.763879,33.093110,4.812001
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,6.086707,18.790775,34.137105,6.151844


In [19]:
#growing season mean
temp90 = temp_growing.groupby("lat_rho").surf_temp.mean().reset_index()
temp90
#Adding this new nitrate metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["Mean_NO3_growing"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_NO3,Mean_NO3,P90_NO3,P10_NO3_growing,Mean_NO3_growing
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,1.959780,12.215722,27.606370,1.247808,15.681467
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,1.980009,11.560737,26.331103,1.072389,14.680380
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,2.470371,14.330551,32.062062,1.715417,19.458192
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,2.549008,15.126274,33.368498,1.709247,20.880819
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,3.473037,14.665780,30.624957,1.826484,17.772663
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,3.545663,14.368735,29.879866,1.907076,17.177929
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,4.526276,15.796427,31.443690,3.330555,19.365914
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,5.512704,17.611699,33.152401,4.562867,21.841788
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,5.833533,17.763879,33.093110,4.812001,22.031183
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,6.086707,18.790775,34.137105,6.151844,23.823527


In [20]:
#Growing season 90th percentile
temp90 = temp_growing.groupby("lat_rho").surf_temp.quantile(q = 0.9).reset_index()
temp90
#Adding this new nitrate metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["P90_NO3_growing"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P10_NO3,Mean_NO3,P90_NO3,P10_NO3_growing,Mean_NO3_growing,P90_NO3_growing
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,1.959780,12.215722,27.606370,1.247808,15.681467,30.605222
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,1.980009,11.560737,26.331103,1.072389,14.680380,29.411907
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,2.470371,14.330551,32.062062,1.715417,19.458192,34.416961
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,2.549008,15.126274,33.368498,1.709247,20.880819,35.046573
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,3.473037,14.665780,30.624957,1.826484,17.772663,33.163644
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,3.545663,14.368735,29.879866,1.907076,17.177929,32.663994
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,4.526276,15.796427,31.443690,3.330555,19.365914,33.653237
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,5.512704,17.611699,33.152401,4.562867,21.841788,34.836845
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,5.833533,17.763879,33.093110,4.812001,22.031183,34.852540
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,6.086707,18.790775,34.137105,6.151844,23.823527,35.608534


In [21]:
sites.to_csv('C:/Users/sarah/Documents/GitHub/ORKA_StatusReport_FollowupPaper/StatusReport_FollowupPaper_Data/MasterSites_WithNO3Metrics_20132023.csv')